# MoshiRAG retrieval-latency sweep on a Colab T4 (low-memory version)

Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Every step is resumable:
if Colab disconnects, rerun the setup cells (0–5) and then the step you were on — completed runs are skipped.

What runs: MoshiRAG (int8-quantized LM, float16), the same frame-based retrieval-latency injection as the
A100 version, a fixed reference whose embedding is precomputed once. No STT, no retrieval LLM.
See `experiments/async_retrieval_latency/t4/README.md`.

## 0. Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 1. Settings
Results and the embedding cache go to Google Drive so they survive disconnects.
The Hugging Face cache (~18 GB, or ~30 GB with the encoder) stays on the local disk.

In [2]:
REPO_URL = "https://github.com/zxx3312/moshi-rag.git"   # your fork
BRANCH = "exp/async-retrieval-latency"
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/moshirag_t4"
else:
    WORK = "/content/moshirag_t4"
os.makedirs(WORK, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_home"   # set before anything imports huggingface_hub
REPO = "/content/moshi-rag"
EXP = f"{REPO}/experiments/async_retrieval_latency"
CACHE = f"{WORK}/reference_cache"
RESULTS = f"{WORK}/results"
print(WORK, CACHE, RESULTS, sep="\n")

Mounted at /content/drive
/content/drive/MyDrive/moshirag_t4
/content/drive/MyDrive/moshirag_t4/reference_cache
/content/drive/MyDrive/moshirag_t4/results


## 2. Clone the repository (private fork: use a GitHub token in the URL or `gh auth login`)

In [3]:
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
!git -C {REPO} log --oneline -1

Cloning into '/content/moshi-rag'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (344/344), done.
remote: Total 400 (delta 26), reused 396 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (400/400), 2.47 MiB | 6.27 MiB/s, done.
Resolving deltas: 100% (26/26), done.
6a44745 (HEAD -> exp/async-retrieval-latency, origin/exp/async-retrieval-latency) Modified: none Added: t4/*


## 3. Install pinned dependencies (2–4 min; nothing here imports torch in this notebook, so no restart is needed)

In [4]:
!pip install -q -r {EXP}/t4/requirements-colab.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip uninstall -y -q torchvision
!python -c "import torch, bitsandbytes, xformers, transformers, sphn, huggingface_hub; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0)); print('bnb', bitsandbytes.__version__, '| xformers', xformers.__version__, '| transformers', transformers.__version__, '| hub', huggingface_hub.__version__)"
!python -c "import torch, bitsandbytes as bnb; from bitsandbytes import functional as F; w=torch.randn(256,128,device='cuda',dtype=torch.float16); CB,SCB,_=F.int8_vectorwise_quant(w); s=bnb.MatmulLtState(); s.CB,s.SCB,s.has_fp16_weights=CB,SCB,False; x=torch.randn(4,128,device='cuda',dtype=torch.float16); y=bnb.matmul(x,CB,state=s); r=x.float()@w.float().T; print('int8 matmul on T4 OK, rel err', float((y.float()-r).norm()/r.norm()))"
!python -c "from transformers import AutoTokenizer; print('transformers import OK')"


torch 2.9.1+cu128 | cuda True Tesla T4
bnb 0.49.2 | xformers 0.0.33.post2 | transformers 4.55.3 | hub 0.36.2
int8 matmul on T4 OK, rel err 0.008307171054184437
transformers import OK


## 4. Hugging Face login
Needed only for step 6 (the reference encoder's tokenizer comes from the gated
`meta-llama/Llama-3.2-3B-Instruct`; only its ~9 MB tokenizer files are downloaded, never the Llama weights).
Use a **read** token whose account has accepted the Llama 3.2 license.

In [12]:
from getpass import getpass
from huggingface_hub import login, whoami
login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)
print("logged in as", whoami()["name"])


Hugging Face read token: ··········
logged in as zhucynthia3312


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. Make the question WAVs (Piper TTS in a separate venv, CPU, ~1 s per question)

In [13]:
import json, pathlib, shutil, subprocess, sys
AUDIO = pathlib.Path(f"{WORK}/audio")   # 存在 Drive 上：只生成一次，--resume 时复用
AUDIO.mkdir(parents=True, exist_ok=True)
link = pathlib.Path(f"{EXP}/configs/audio")
if link.is_dir() and not link.is_symlink():      # 旧版单元格生成的 WAV：保留下来
    for f in link.glob("*.wav"):
        if not (AUDIO / f.name).exists():
            shutil.move(str(f), AUDIO / f.name)
    shutil.rmtree(link)
if not link.is_symlink():
    link.symlink_to(AUDIO, target_is_directory=True)

# Piper 装进单独的文件夹（不用 venv：Colab 的 python 常常缺少 ensurepip）
TTS = "/content/tts_pkgs"
VOICE = "/content/piper_voices/en_US-lessac-medium.onnx"
if not os.path.isdir(f"{TTS}/piper"):
    !pip install -q --target {TTS} piper-tts
env = dict(os.environ, PYTHONPATH=TTS)

def run(cmd):
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout, r.stderr)
        raise RuntimeError(f"failed: {' '.join(cmd)}")
    return r.stdout

if not os.path.exists(VOICE):
    run([sys.executable, "-m", "piper.download_voices", "en_US-lessac-medium", "--data-dir", "/content/piper_voices"])
for manifest in [f"{EXP}/configs/samples.jsonl", f"{EXP}/t4/configs/samples_small.jsonl"]:
    for line in open(manifest):
        s = json.loads(line)
        wav = (pathlib.Path(manifest).parent / s["wav"]).resolve()
        if not wav.exists():
            print(run([sys.executable, f"{EXP}/scripts/make_tts_question.py", "--voice", VOICE,
                       "--text", s["question"], "--out", str(wav)]).strip())
!ls -la {AUDIO}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.12 requires torchvision>=0.11, which is not installed.
timm 1.0.30 requires torchvision, which is not installed.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
langgraph-sdk 0.4.5 requires websockets<17,>=14, but you have websockets 17.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36

In [14]:
!python -c "import numpy, torch; print('numpy', numpy.__version__, numpy.__file__); print('torch', torch.__version__)"


numpy 2.2.6 /usr/local/lib/python3.13/dist-packages/numpy/__init__.py
torch 2.9.1+cu128


## 6. Precompute reference embeddings (one time; separate process, frees the GPU when it exits)
Downloads the MoshiRAG checkpoint (15.4 GB) and the ARC encoder (12.1 GB). On a T4 the encoder autocasts to
float16 (no bf16 support); the official encoder uses bfloat16. For exact parity, compute the cache on an
A100 instead and copy it to `CACHE`.

In [18]:
p = f"{EXP}/t4/precompute_reference_embeddings.py"
s = open(p).read()
anchor = '    logger.info("ARC weights assigned (%d unexpected keys ignored)", len(result.unexpected_keys))\n'
if "cond.requires_grad_(False)" not in s:
    assert anchor in s, "anchor not found"
    s = s.replace(anchor, anchor + "    cond.requires_grad_(False)\n")
    open(p, "w").write(s)
print("patched" if "cond.requires_grad_(False)" in open(p).read() else "NOT patched")


patched


In [19]:
!python {EXP}/t4/precompute_reference_embeddings.py --manifest {EXP}/configs/samples.jsonl {EXP}/t4/configs/samples_small.jsonl --cache-dir {CACHE} --overwrite


00:32:56 INFO 5 distinct references, 5 to encode, cache: /content/drive/MyDrive/moshirag_t4/reference_cache
00:33:02 INFO NumExpr defaulting to 2 threads.
I0000 00:00:1791246784.224753   16681 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
00:33:10 INFO Loading tokenizer from meta-llama/Llama-3.2-3B-Instruct
00:33:59 INFO ARC weights assigned (2 unexpected keys ignored)
00:33:59 INFO encoder loaded in 61s; GPU allocated 12.17 GB (peak 12.21 GB)
00:34:00 INFO cached ref_64497a4c90128d2d.safetensors: shape (9, 4096) (Angel Falls in Venezuela is the tallest waterfall in the wor)
00:34:00 INFO cached ref_34f31f73306cf092.safetensors: shape (8, 4096) (Canberra is the capital city of Australia. It was built as a)
00:34:00 INFO cached ref_a6b1cfd3df3534f7.safetensors: shape (7, 

## 7. Step 1 — 1 sample × 1 delay (3 s)
Checks: model loads in 16 GB, `[RET]` → injection works, audio + JSONL are written. Read the memory lines in the output.

In [20]:
ARGS = f"--reference-cache {CACHE} --output-root {RESULTS} --resume"
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS} --dry-run
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS}

T4 DRY RUN: no torch import, no model loading, no GPU use

[manifest] /content/moshi-rag/experiments/async_retrieval_latency/configs/samples.jsonl (1/1 samples selected)
  - tallest_waterfall: 3.868 s audio | reference cached [9, 4096] torch.float32 autocast=float16

[delays] condition  steps effective
                 3     38      3.04

[resume] /content/drive/MyDrive/moshirag_t4/results/step1_1x1: {'records': 0, 'corrupt_lines_dropped': 0, 'failed_to_retry': 0}

[plan] 1 runs (1 samples x 1 seeds x 1 delays), 0 already done, 1 to run
  run dir: /content/drive/MyDrive/moshirag_t4/results/step1_1x1
  model: int8 (q8) float16, CUDA graphs off, STT off, retrieval LLM off, reference encoder off (cache)

  confirmed: torch was not imported

T4 DRY RUN PASSED (0 errors, 0 warnings)
00:37:14 INFO latency_sweep_t4: run directory /content/drive/MyDrive/moshirag_t4/results/step1_1x1 (resumed); 1 planned, 0 done, 1 to run
00:37:17 INFO gpu_memory: [GPU] process_start                allocated   

In [21]:
import json
run = f"{RESULTS}/step1_1x1"
print(open(f"{run}/run_status.json").read())
for r in map(json.loads, open(f"{run}/results.jsonl")):
    print({k: r[k] for k in ["run_id", "status", "stop_reason", "rag_triggered", "retrieval_trigger_time",
                             "reference_injection_time", "trigger_to_injection_s", "gpu_peak_allocated_mb", "run_wall_s"]})
    print("pre-RAG :", r["pre_rag_transcript"]); print("post-RAG:", r["post_rag_transcript"])
from IPython.display import Audio
Audio(f"{run}/audio/" + r["run_id"] + "_stereo.wav")

{
  "status": "finished",
  "counts": {
    "ok": 1
  },
  "planned_runs": 1,
  "remaining_runs": [],
  "updated_at": "2026-10-06T00:40:19+00:00"
}
{'run_id': 'tallest_waterfall__d3s__s42424242', 'status': 'ok', 'stop_reason': 'tail_silence', 'rag_triggered': True, 'retrieval_trigger_time': 3.12, 'reference_injection_time': 6.64, 'trigger_to_injection_s': 3.52, 'gpu_peak_allocated_mb': 10798.1, 'run_wall_s': 75.2}
pre-RAG : Oh, that's a good question! Angel Falls in Venezuela
post-RAG: is the tallest. It's nine hundred seventy-nine meters high, with an eight hundred seven meter uninterrupted drop. Pretty impressive, right?


## 8. Step 2 — 1 sample × all delays (0, 1, 2, 3, 5, 8 s)

In [22]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --run-name step2_1xall {ARGS}

00:42:18 INFO latency_sweep_t4: run directory /content/drive/MyDrive/moshirag_t4/results/step2_1xall (new); 6 planned, 0 done, 6 to run
00:42:20 INFO gpu_memory: [GPU] process_start                allocated      0.0 MB | reserved      0.0 MB | peak      0.0 MB | device    104.9 / 14913 MB
00:42:24 INFO gpu_memory: [GPU] mimi_loaded                  allocated    373.3 MB | reserved    766.0 MB | peak    748.6 MB | device    886.9 / 14913 MB
00:42:27 INFO numexpr.utils: NumExpr defaulting to 2 threads.
I0000 00:00:1791247348.817987   19415 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
00:42:36 INFO gpu_memory: [GPU] lm_skeleton_on_meta          allocated    373.3 MB | reserved    766.0 MB | peak    748.6 MB | device    886.9 / 14913 MB
00:42:36 INFO q8_moshirag: q8 loader:

In [23]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step2_1xall

medians in seconds unless noted; rates over ok runs with a defined value
        delay             ok           RET%       ret->inj          onset       1st info  preRAG speech   preRAG words        max gap       ans<inj%      rep.fill%
          0.0              1            1.0           0.48           0.32            2.8           0.48            1.0           0.56            0.0            0.0
          1.0              1            1.0           1.52           0.32           2.96            1.2            5.0           1.76            0.0            0.0
          2.0              1            1.0           2.48           0.32           3.04           1.84            6.0           1.68            0.0            0.0
          3.0              1            1.0           3.52           0.32           3.04            2.8            9.0           0.72            1.0            0.0
          5.0              1            1.0           5.52           0.32           3.04           4.48    

## 9. Step 3 — small multi-sample sweep (5 samples × 6 delays = 30 runs)
Long: rerun this cell after a disconnect, it resumes. The 5 questions are examples; choose your own evaluation set.

In [24]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/t4/configs/samples_small.jsonl --run-name step3_small {ARGS}

流式输出内容被截断，只能显示最后 5000 行内容。
01:04:42 INFO moshi.inference_utils.turn_manager: [Buffered Model] 'Melbourne'
01:04:43 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 293.8ms
01:04:43 INFO moshi.inference_utils.turn_manager: [Buffered Model] '.'
01:04:43 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 299.1ms
01:04:43 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 294.6ms
01:04:44 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 283.3ms
01:04:44 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 307.4ms
01:04:44 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 288.3ms
01:04:45 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 290.9ms
01:04:45 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 285.0ms
01:04:45 WARNING moshi.inference_utils.batch_runner: batched step (1/1 active) took 295

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step3_small

## GPU memory log

In [ ]:
for line in open(f"{RESULTS}/step1_1x1/logs/gpu_memory.jsonl"):
    r = json.loads(line)
    print(f"{r['event']:28s} allocated {r.get('allocated_mb', 0):8.0f} MB  peak {r.get('peak_allocated_mb', 0):8.0f} MB  device {r.get('device_used_mb', 0):8.0f} MB")

In [25]:
!git -C {REPO} status --short
!git -C {REPO} diff --stat


 M experiments/async_retrieval_latency/t4/precompute_reference_embeddings.py
?? experiments/async_retrieval_latency/configs/audio
 .../async_retrieval_latency/t4/precompute_reference_embeddings.py        | 1 +
 1 file changed, 1 insertion(+)
